
# PH3043 — Reproducibility and Scientific Validation

## 1. Opposite-Charge Selection and Event Cut-Flow

### Objective

Verify that the CERN SingleMu and DoubleMu datasets contain
oppositely charged muon pairs, as required by the project brief.

The condition for an opposite-charge pair is:

$$
Q_1Q_2=-1
$$

The analysis will document:
- Total input events
- Missing or nonfinite kinematic values
- Opposite-charge and same-charge events
- Duplicate collision identifiers
- Events within the baseline Z-resonance fitting window

The original datasets will not be modified.


In [1]:
#Run the charge and cutflow anlysis

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

files = {
    "SingleMu": RAW_DIR / "Dimuon_SingleMu.csv",
    "DoubleMu": RAW_DIR / "Dimuon_DoubleMu.csv"
}

required = [
    "Run", "Event", "Q1", "Q2",
    "pt1", "eta1", "phi1",
    "pt2", "eta2", "phi2"
]

results = []
datasets = {}

for name, path in files.items():

    df = pd.read_csv(path)
    datasets[name] = df

    # Check required columns
    missing_columns = set(required) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{name}: missing columns {missing_columns}"
        )

    # Missing / nonfinite kinematic values
    kinematics = [
        "pt1", "eta1", "phi1",
        "pt2", "eta2", "phi2"
    ]

    finite = np.isfinite(
        df[kinematics].to_numpy(dtype=float)
    ).all(axis=1)

    physical_pt = (
        (df["pt1"] >= 0) &
        (df["pt2"] >= 0)
    ).to_numpy()

    valid = finite & physical_pt

    # Opposite-charge selection
    charge_product = df["Q1"] * df["Q2"]

    opposite = charge_product.eq(-1)
    same = charge_product.eq(1)

    # Inspect stored CERN mass
    mass = pd.to_numeric(df["M"], errors="coerce")

    in_window = (
        mass.ge(85) & mass.lt(97.5)
    )

    # Duplicate identifiers
    duplicates = df.duplicated(
        ["Run", "Event"]
    )

    results.append({
        "Dataset": name,
        "Total rows": len(df),
        "Invalid kinematics": int((~valid).sum()),
        "Opposite charge": int(opposite.sum()),
        "Same charge": int(same.sum()),
        "Other charge products": int(
            (~opposite & ~same).sum()
        ),
        "Duplicate IDs": int(duplicates.sum()),
        "Valid opposite-charge events": int(
            (valid & opposite.to_numpy()).sum()
        ),
        "Stored M in fit window": int(
            (valid & opposite.to_numpy()
             & in_window.to_numpy()).sum()
        )
    })

cutflow = pd.DataFrame(results)

display(cutflow)


,Dataset,Total rows,Invalid kinematics,Opposite charge,Same charge,Other charge products,Duplicate IDs,Valid opposite-charge events,Stored M in fit window
0,SingleMu,83761,0,83761,0,0,0,83761,10179
1,DoubleMu,100000,0,100000,0,0,0,100000,4688


In [3]:
#save the results

# Save cut-flow table

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cutflow.to_csv(
    RESULTS_DIR / "charge_cutflow.csv",
    index=False
)

print("Saved charge_cutflow.csv")


Saved charge_cutflow.csv


## 2. Dataset Provenance and File Integrity

### Objective

Document the origin, retrieval information, file properties
and cryptographic SHA-256 checksums of the two CERN datasets.

The checksums provide reproducible identifiers for the exact
files used in the analysis.

The original CSV files will be retained unchanged.

A checksum verifies file identity, but does not by itself
establish physical data quality or correctness.

In [6]:

import hashlib
from pathlib import Path
import pandas as pd
from datetime import date

def calculate_sha256(path):
    sha = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            sha.update(block)

    return sha.hexdigest()


dataset_info = [
    {
        "Dataset": "SingleMu",
        "CERN record": "cms-5202",
        "URL": "https://opendata.cern.ch/record/cms-5202",
        "Filename": "Dimuon_SingleMu.csv",
        "File": RAW_DIR / "Dimuon_SingleMu.csv"
    },
    {
        "Dataset": "DoubleMu",
        "CERN record": "cms-5201",
        "URL": "https://opendata.cern.ch/record/cms-5201",
        "Filename": "Dimuon_DoubleMu.csv",
        "File": RAW_DIR / "Dimuon_DoubleMu.csv"
    }
]

provenance_rows = []

for info in dataset_info:

    path = info["File"]

    if not path.exists():
        raise FileNotFoundError(path)

    provenance_rows.append({
        "Dataset": info["Dataset"],
        "CERN record": info["CERN record"],
        "Source URL": info["URL"],
        "Filename": info["Filename"],
        "Size (MB)": path.stat().st_size / (1024**2),
        "SHA-256": calculate_sha256(path),
        "Download date": "2026-11-17",
        "Momentum-scale correction applied":
            "None in student analysis; upstream status unknown"
    })

provenance = pd.DataFrame(provenance_rows)

display(provenance.T)


,0,1
Dataset,SingleMu,DoubleMu
CERN record,cms-5202,cms-5201
Source URL,https://opendata.cern.ch/record/cms-5202,https://opendata.cern.ch/record/cms-5201
Filename,Dimuon_SingleMu.csv,Dimuon_DoubleMu.csv
Size (MB),11.259941,13.290253
SHA-256,9901450546336067eebb18e8294ed56cb32d583193d097...,58caa45c580f8bcf4b457281c8525a56de0e0b8eb0e0b2...
Download date,2026-11-17,2026-11-17
Momentum-scale correction applied,None in student analysis; upstream status unknown,None in student analysis; upstream status unknown


In [8]:

column_definitions = pd.DataFrame([
    ("Run", "CMS data-taking run identifier", "dimensionless"),
    ("Event", "Collision event identifier", "dimensionless"),
    ("type1/type2", "Muon reconstruction category", "categorical"),
    ("E1/E2", "Muon energy", "GeV"),
    ("px1/px2", "Momentum x-component", "GeV"),
    ("py1/py2", "Momentum y-component", "GeV"),
    ("pz1/pz2", "Momentum z-component", "GeV"),
    ("pt1/pt2", "Transverse momentum", "GeV"),
    ("eta1/eta2", "Pseudorapidity", "dimensionless"),
    ("phi1/phi2", "Azimuthal angle", "radians"),
    ("Q1/Q2", "Muon electric charge sign", "units of e"),
    ("M", "Supplied dimuon invariant mass", "GeV")
], columns=[
    "Variable",
    "Description",
    "Unit"
])

display(column_definitions)

column_definitions.to_csv(
    RESULTS_DIR / "column_definitions.csv",
    index=False
)


,Variable,Description,Unit
0,Run,CMS data-taking run identifier,dimensionless
1,Event,Collision event identifier,dimensionless
2,type1/type2,Muon reconstruction category,categorical
3,E1/E2,Muon energy,GeV
4,px1/px2,Momentum x-component,GeV
5,py1/py2,Momentum y-component,GeV
6,pz1/pz2,Momentum z-component,GeV
7,pt1/pt2,Transverse momentum,GeV
8,eta1/eta2,Pseudorapidity,dimensionless
9,phi1/phi2,Azimuthal angle,radians


In [10]:

provenance.to_csv(
    RESULTS_DIR / "dataset_provenance.csv",
    index=False
)

print("Saved dataset_provenance.csv")
print("Saved column_definitions.csv")


Saved dataset_provenance.csv
Saved column_definitions.csv



## 3. Physics Unit Tests for Invariant-Mass Reconstruction

### Objective

Verify that the four-momentum reconstruction correctly
recovers a known parent-particle mass.

Three tests are performed:

1. Two back-to-back muons from a parent at rest.
2. The same decay after applying a Lorentz boost.
3. Agreement between Cartesian and pT-eta-phi reconstruction.

### Physical principle

The invariant mass is calculated using:

$$
M^2 = (E_1+E_2)^2 -
(p_{x1}+p_{x2})^2 -
(p_{y1}+p_{y2})^2 -
(p_{z1}+p_{z2})^2
$$

Natural units are used, with c = 1.

The invariant mass must remain unchanged under a Lorentz
transformation, apart from floating-point rounding.


In [13]:
#resuable mass function

import numpy as np

MUON_MASS = 0.105658  # GeV

def invariant_mass_cartesian(p1, p2):
    """
    Calculate invariant mass from two four-vectors.

    Four-vector order: [E, px, py, pz]
    Units: GeV
    """
    total = np.asarray(p1, dtype=float) + np.asarray(p2, dtype=float)

    E, px, py, pz = total

    m2 = E**2 - px**2 - py**2 - pz**2

    if m2 < -1e-9:
        raise ValueError("Unphysical negative invariant mass squared")

    return np.sqrt(max(m2, 0.0))


def four_vector_from_pt_eta_phi(pt, eta, phi, mass=MUON_MASS):
    """Construct a muon four-vector from cylindrical coordinates."""

    px = pt * np.cos(phi)
    py = pt * np.sin(phi)
    pz = pt * np.sinh(eta)

    E = np.sqrt(px**2 + py**2 + pz**2 + mass**2)

    return np.array([E, px, py, pz])


In [15]:

# Test 1: Parent particle at rest

KNOWN_MASS = 91.2  # GeV

E_mu = KNOWN_MASS / 2

p_mu = np.sqrt(E_mu**2 - MUON_MASS**2)

muon_1 = np.array([E_mu, p_mu, 0.0, 0.0])
muon_2 = np.array([E_mu, -p_mu, 0.0, 0.0])

reconstructed = invariant_mass_cartesian(muon_1, muon_2)

print("TEST 1 — PARENT AT REST")
print(f"Injected mass: {KNOWN_MASS:.9f} GeV")
print(f"Recovered mass: {reconstructed:.9f} GeV")
print(f"Absolute error: {abs(reconstructed-KNOWN_MASS):.3e} GeV")

np.testing.assert_allclose(
    reconstructed,
    KNOWN_MASS,
    rtol=1e-12,
    atol=1e-10
)

print("PASS: Known mass recovered")


TEST 1 — PARENT AT REST
Injected mass: 91.200000000 GeV
Recovered mass: 91.200000000 GeV
Absolute error: 0.000e+00 GeV
PASS: Known mass recovered


In [17]:

# Test 2: Lorentz boost invariance

beta = 0.6
gamma_lorentz = 1 / np.sqrt(1 - beta**2)

def boost_z(four_vector, beta):
    E, px, py, pz = four_vector

    gamma = 1 / np.sqrt(1-beta**2)

    return np.array([
        gamma * (E + beta*pz),
        px,
        py,
        gamma * (pz + beta*E)
    ])

muon_1_boosted = boost_z(muon_1, beta)
muon_2_boosted = boost_z(muon_2, beta)

boosted_mass = invariant_mass_cartesian(
    muon_1_boosted,
    muon_2_boosted
)

print("TEST 2 — LORENTZ BOOST")

print(f"Original mass: {reconstructed:.9f} GeV")
print(f"Boosted mass: {boosted_mass:.9f} GeV")
print(f"Absolute error: {abs(boosted_mass-KNOWN_MASS):.3e} GeV")

np.testing.assert_allclose(
    boosted_mass,
    KNOWN_MASS,
    rtol=1e-12,
    atol=1e-10
)

print("PASS: Invariant mass unchanged under boost")


TEST 2 — LORENTZ BOOST
Original mass: 91.200000000 GeV
Boosted mass: 91.200000000 GeV
Absolute error: 1.421e-14 GeV
PASS: Invariant mass unchanged under boost


In [19]:
#compare coodinate reoresntation

# Test 3: Cartesian versus pT-eta-phi reconstruction

def cartesian_to_pt_eta_phi(four_vector):
    E, px, py, pz = four_vector

    pt = np.hypot(px, py)

    if pt <= 0:
        raise ValueError("pT must be positive for this test")

    eta = np.arcsinh(pz / pt)
    phi = np.arctan2(py, px)

    return pt, eta, phi


pt1, eta1, phi1 = cartesian_to_pt_eta_phi(muon_1_boosted)
pt2, eta2, phi2 = cartesian_to_pt_eta_phi(muon_2_boosted)

reco_1 = four_vector_from_pt_eta_phi(pt1, eta1, phi1)
reco_2 = four_vector_from_pt_eta_phi(pt2, eta2, phi2)

cylindrical_mass = invariant_mass_cartesian(reco_1, reco_2)

print("TEST 3 — COORDINATE CONSISTENCY")

print(f"Cartesian mass: {boosted_mass:.9f} GeV")
print(f"pT-eta-phi mass: {cylindrical_mass:.9f} GeV")

np.testing.assert_allclose(
    cylindrical_mass,
    boosted_mass,
    rtol=1e-12,
    atol=1e-10
)

print("PASS: Both reconstruction methods agree")


TEST 3 — COORDINATE CONSISTENCY
Cartesian mass: 91.200000000 GeV
pT-eta-phi mass: 91.200000000 GeV
PASS: Both reconstruction methods agree



## 4. DoubleMu Mass-Reconstruction Discrepancy Audit

### Objective

Investigate events where the independently reconstructed
dimuon invariant mass differs substantially from the
mass stored in the CERN dataset.

### Questions

1. How many events have mass discrepancies above
   0.1, 1 and 10 GeV?
2. How many discrepant events fall within the baseline
   Z-resonance fitting window?
3. Are discrepancies associated with inconsistencies
   between stored and reconstructed transverse momenta?
4. Do these events require additional validation before
   the final analysis?

### Scientific approach

All events are initially retained. No event is removed
solely because its reconstructed mass differs from the
stored CERN mass.


In [22]:

# Section 4 — DoubleMu mass discrepancy analysis

import numpy as np
import pandas as pd

double_audit = datasets["DoubleMu"].copy()

def reconstruct_pair_mass(data):
    pt1 = data["pt1"].to_numpy(dtype=float)
    pt2 = data["pt2"].to_numpy(dtype=float)
    eta1 = data["eta1"].to_numpy(dtype=float)
    eta2 = data["eta2"].to_numpy(dtype=float)
    phi1 = data["phi1"].to_numpy(dtype=float)
    phi2 = data["phi2"].to_numpy(dtype=float)

    # Each muon's reconstructed four-momentum
    px1 = pt1 * np.cos(phi1)
    py1 = pt1 * np.sin(phi1)
    pz1 = pt1 * np.sinh(eta1)

    px2 = pt2 * np.cos(phi2)
    py2 = pt2 * np.sin(phi2)
    pz2 = pt2 * np.sinh(eta2)

    E1 = np.sqrt(
        pt1**2 * np.cosh(eta1)**2 + MUON_MASS**2
    )
    E2 = np.sqrt(
        pt2**2 * np.cosh(eta2)**2 + MUON_MASS**2
    )

    m2 = (
        (E1 + E2)**2
        - (px1 + px2)**2
        - (py1 + py2)**2
        - (pz1 + pz2)**2
    )

    return np.where(
        np.isfinite(m2) & (m2 >= 0),
        np.sqrt(np.maximum(m2, 0)),
        np.nan
    )

double_audit["M_reco"] = reconstruct_pair_mass(double_audit)

double_audit["M_diff"] = (
    double_audit["M_reco"] - double_audit["M"]
).abs()

# Compare two transverse-momentum representations
for i in [1, 2]:
    double_audit[f"pt{i}_diff"] = (
        double_audit[f"pt{i}"]
        - np.hypot(
            double_audit[f"px{i}"],
            double_audit[f"py{i}"]
        )
    ).abs()

double_audit["pt_flag"] = (
    (double_audit["pt1_diff"] > 0.01)
    | (double_audit["pt2_diff"] > 0.01)
)

stored_window = (
    double_audit["M"].ge(85)
    & double_audit["M"].lt(97.5)
)

reco_window = (
    double_audit["M_reco"].ge(85)
    & double_audit["M_reco"].lt(97.5)
)

rows = []

for threshold in [0.1, 1.0, 10.0]:
    flag = double_audit["M_diff"] > threshold

    rows.append({
        "Mass difference threshold (GeV)": threshold,
        "Total flagged": int(flag.sum()),
        "Stored M in fit window":
            int((flag & stored_window).sum()),
        "Reconstructed M in fit window":
            int((flag & reco_window).sum()),
        "Flagged with pT inconsistency":
            int((flag & double_audit["pt_flag"]).sum())
    })

discrepancy_summary = pd.DataFrame(rows)

display(discrepancy_summary)

print("\nLargest mass discrepancies:")

columns = [
    "Run", "Event", "M", "M_reco", "M_diff",
    "pt1_diff", "pt2_diff", "type1", "type2"
]

display(
    double_audit.nlargest(10, "M_diff")[columns]
)

print("\nWindow membership disagreement:")
print("Stored mass only:",
      int((stored_window & ~reco_window).sum()))
print("Reconstructed mass only:",
      int((reco_window & ~stored_window).sum()))


,Mass difference threshold (GeV),Total flagged,Stored M in fit window,Reconstructed M in fit window,Flagged with pT inconsistency
0,0.1,2445,27,27,2444
1,1.0,370,11,11,369
2,10.0,29,2,2,29



Largest mass discrepancies:


,Run,Event,M,M_reco,M_diff,pt1_diff,pt2_diff,type1,type2
529,165617,80284893,19.6952,91.336886,71.641686,0.000034,504.239864,G,G
92314,166895,115064722,28.3973,83.529725,55.132425,0.000019,214.892672,G,G
60460,173430,217669832,74.8259,29.375809,45.450091,0.000032,93.032843,G,G
1183,165617,85137198,94.6790,137.968609,43.289609,45.979832,0.000052,G,G
30378,163589,47704774,49.9828,88.975925,38.993125,0.000020,117.900799,G,G
15480,166784,41404601,23.5155,57.219776,33.704276,0.000051,237.488983,G,G
26128,163589,27043076,17.0089,47.764372,30.755472,18.804523,0.000018,G,G
6771,165617,137061447,145.1190,120.343364,24.775636,0.000045,12.644176,G,G
16540,166784,50285782,25.8176,2.203389,23.614211,4762.091021,0.000072,G,G
30979,163589,50276168,54.7612,78.168230,23.407030,24.176114,0.000021,G,G



Window membership disagreement:
Stored mass only: 2
Reconstructed mass only: 2


In [24]:

discrepancy_summary.to_csv(
    RESULTS_DIR / "doublemu_discrepancy_summary.csv",
    index=False
)

double_audit.nlargest(
    20, "M_diff"
)[columns].to_csv(
    RESULTS_DIR / "doublemu_largest_mass_discrepancies.csv",
    index=False
)

print("DoubleMu discrepancy audit files saved.")


DoubleMu discrepancy audit files saved.



## 5. Fitting-Window Membership and Reconstruction Consistency

### Objective

Identify events whose inclusion in the 85–97.5 GeV
resonance window depends on whether the supplied CERN
mass or independently reconstructed mass is used.

Inspect the recorded four-momenta and transverse-momentum
inconsistencies for these events.

### Scientific decision

No events are removed automatically.

The primary reconstruction method is retained, and
any exclusion is evaluated separately as a sensitivity
analysis using a predefined consistency criterion.


In [29]:
#Examine boundry 4 changing events

# Identify events selected differently by stored
# and independently reconstructed invariant mass

stored_window = double_audit["M"].between(
    85, 97.5, inclusive="left"
)

reco_window = double_audit["M_reco"].between(
    85, 97.5, inclusive="left"
)

different_membership = stored_window != reco_window

boundary_events = double_audit.loc[
    different_membership
].copy()

boundary_events["Selection"] = np.select(
    [
        stored_window.loc[boundary_events.index],
        reco_window.loc[boundary_events.index]
    ],
    [
        "Stored mass only",
        "Reconstructed mass only"
    ],
    default="Neither"
)

columns_to_show = [
    "Run", "Event", "Selection",
    "M", "M_reco", "M_diff",
    "pt1", "pt1_diff",
    "pt2", "pt2_diff",
    "eta1", "eta2"
]

print(
    "Events changing fit-window membership:",
    len(boundary_events)
)

display(
    boundary_events[columns_to_show].sort_values(
        "M_diff", ascending=False
    )
)

# Save for reproducibility
boundary_events[columns_to_show].to_csv(
    RESULTS_DIR / "doublemu_window_boundary_events.csv",
    index=False
)


Events changing fit-window membership: 4


,Run,Event,Selection,M,M_reco,M_diff,pt1,pt1_diff,pt2,pt2_diff,eta1,eta2
529,165617,80284893,Reconstructed mass only,19.6952,91.336886,71.641686,3.4398,0.000034,528.434,504.239864,-0.7705,-0.0208
1183,165617,85137198,Stored mass only,94.6790,137.968609,43.289609,86.8798,45.979832,12.307,0.000052,-1.6893,1.1149
30378,163589,47704774,Reconstructed mass only,49.9828,88.975925,38.993125,7.3991,0.000020,172.303,117.900799,1.4175,0.0418
44821,166033,486156267,Stored mass only,90.7423,79.131790,11.610510,204.6980,64.383320,177.920,0.000908,0.1903,0.3997


In [38]:

# Audit the actual reconstructed-mass fitting sample

z_events = double_audit.loc[reco_window].copy()

thresholds = [0.01, 0.1, 1.0, 10.0]

rows = []

for threshold in thresholds:
    affected = z_events["M_diff"] > threshold

    rows.append({
        "Threshold (GeV)": threshold,
        "Affected events": int(affected.sum()),
        "Affected (%)": 100 * affected.mean(),
        "With pT inconsistency": int(
            (affected & z_events["pt_flag"]).sum()
        )
    })

z_discrepancy_table = pd.DataFrame(rows)

print("Total reconstructed Z-window events:", len(z_events))
display(z_discrepancy_table.round(4))

print("\nLargest discrepancies within fit window:")

display(
    z_events.nlargest(10, "M_diff")[
        [
            "Run", "Event", "M", "M_reco",
            "M_diff", "pt1_diff", "pt2_diff"
        ]
    ]
)

z_discrepancy_table.to_csv(
    RESULTS_DIR / "doublemu_z_window_discrepancies.csv",
    index=False
)


Total reconstructed Z-window events: 4688


,Threshold (GeV),Affected events,Affected (%),With pT inconsistency
0,0.01,28,0.5973,28
1,0.10,27,0.5759,27
2,1.00,11,0.2346,11
3,10.00,2,0.0427,2



Largest discrepancies within fit window:


,Run,Event,M,M_reco,M_diff,pt1_diff,pt2_diff
529,165617,80284893,19.6952,91.336886,71.641686,0.000034,504.239864
30378,163589,47704774,49.9828,88.975925,38.993125,0.000020,117.900799
18081,166784,64955408,89.3498,85.482852,3.866948,2.992562,0.000007
73307,173692,566039031,96.3707,94.406683,1.964017,0.000605,0.575555
60168,173430,212904037,96.5190,94.692818,1.826182,2.270732,0.000035
16927,166784,55061433,90.1835,88.443836,1.739664,0.000024,1.889794
33698,160957,110648193,91.5107,90.065423,1.445277,0.000037,0.609462
63900,167807,1098135642,92.2303,90.797618,1.432682,0.000044,1.280264
87500,166895,77609830,92.6226,91.193973,1.428627,3.179032,0.114578
10415,166701,683464232,86.8296,85.417743,1.411857,0.000011,1.338102


## 6. Python Environment and Dependency Versions

In [34]:

# Record exact versions used for this analysis

import sys
import platform
import importlib.metadata as metadata

packages = [
    "numpy",
    "pandas",
    "scipy",
    "matplotlib",
    "jupyter",
    "ipykernel",
    "nbformat"
]

print("Python:", sys.version)
print("Operating system:", platform.platform())

versions = {}

for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

print("\nPACKAGE VERSIONS")

for package, version in versions.items():
    print(f"{package}=={version}")

# Save the tested package versions
requirements_path = PROJECT_ROOT / "requirements.txt"

with open(requirements_path, "w") as f:
    for package, version in versions.items():
        f.write(f"{package}=={version}\n")

print("\nSaved:", requirements_path)


Python: 3.11.18 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 18:13:29) [MSC v.1942 64 bit (AMD64)]
Operating system: Windows-10-10.0.26300-SP0
jupyter: not installed

PACKAGE VERSIONS
numpy==2.4.6
pandas==3.0.6
scipy==1.17.1
matplotlib==3.11.2
ipykernel==7.3.0
nbformat==5.11.1

Saved: C:\Users\Asini Perera\CP Lab 1\CP Lab 2\PH3043_ZBoson\requirements.txt


NameError: name 'datasets_fixed' is not defined